# CycleGAN (Unpaired Domain Translation) — Detailed Recap + Math + Worked Toy Example

This notebook explains **CycleGAN** step-by-step (intuition → theory → math → runnable toy experiment).

You can run this directly in **Google Colab**.

**What you’ll learn**
- What “unpaired domain translation” means
- Why vanilla GAN loss is not enough for translation
- The CycleGAN architecture: two generators + two discriminators
- The key losses: adversarial, cycle-consistency, (optional) identity
- A runnable toy CycleGAN that translates between two 2D domains with **unpaired** samples


## 1) Problem setup: unpaired domain translation

We have two datasets (two **domains**):
- Domain **X**: samples $x \sim p_X(x)$
- Domain **Y**: samples $y \sim p_Y(y)$

But we **do not** have paired examples $(x, y)$ where $y$ is the correct translation of $x$.

Example: many horse photos (domain X) and many zebra photos (domain Y), but no exact horse↔zebra pairs.

**Goal:** learn a mapping from X to Y and from Y to X:

$$G: X \to Y, \quad F: Y \to X$$

Such that $G(x)$ looks like domain Y samples, and $F(y)$ looks like domain X samples.


## 2) Why a vanilla GAN loss alone fails for translation

Suppose we only learn $G: X\to Y$ with a GAN loss so that $G(x)$ looks like a real $y$.

A discriminator $D_Y$ is trained to distinguish:
- real $y \sim p_Y$
- fake $\hat{y} = G(x)$

Adversarial loss (one common form):

$$\mathcal{L}_{GAN}(G, D_Y, X, Y)
= \mathbb{E}_{y\sim p_Y}[\log D_Y(y)] + \mathbb{E}_{x\sim p_X}[\log(1 - D_Y(G(x)))].$$

**Problem:** This does not force $G(x)$ to preserve the content of $x$.
It could ignore $x$ and output any zebra-looking image that fools $D_Y$.

So we need an extra constraint that ties output back to input.


## 3) The CycleGAN idea: cycle consistency

CycleGAN introduces a second generator $F: Y\to X$ and enforces that going **there and back** recovers the original:

$$F(G(x)) \approx x \quad \text{for } x\sim p_X$$
$$G(F(y)) \approx y \quad \text{for } y\sim p_Y$$

This is called **cycle-consistency**.

Cycle-consistency loss (typically L1):

$$\mathcal{L}_{cyc}(G,F)
= \mathbb{E}_{x\sim p_X}[\|F(G(x)) - x\|_1] + \mathbb{E}_{y\sim p_Y}[\|G(F(y)) - y\|_1].$$

Intuition: if $G$ changes a horse into a zebra, $F$ must be able to change that zebra back into the **same horse**.


## 4) Full CycleGAN architecture

CycleGAN uses **two generators** and **two discriminators**:

- Generators:
  - $G: X\to Y$
  - $F: Y\to X$

- Discriminators:
  - $D_Y$: real vs fake in domain Y
  - $D_X$: real vs fake in domain X

So there are two GAN games:
- $G$ vs $D_Y$
- $F$ vs $D_X$


## 5) Losses used in CycleGAN

### 5.1 Adversarial losses (two directions)

$$\mathcal{L}_{GAN}(G, D_Y, X, Y)
= \mathbb{E}_{y\sim p_Y}[\log D_Y(y)] + \mathbb{E}_{x\sim p_X}[\log(1 - D_Y(G(x)))].$$

$$\mathcal{L}_{GAN}(F, D_X, Y, X)
= \mathbb{E}_{x\sim p_X}[\log D_X(x)] + \mathbb{E}_{y\sim p_Y}[\log(1 - D_X(F(y)))].$$

### 5.2 Cycle-consistency loss

$$\mathcal{L}_{cyc}(G,F)
= \mathbb{E}_{x\sim p_X}[\|F(G(x)) - x\|_1] + \mathbb{E}_{y\sim p_Y}[\|G(F(y)) - y\|_1].$$

### 5.3 (Optional) Identity loss

Often added to preserve color/structure when input is already in the target domain:

$$\mathcal{L}_{id}(G,F)
= \mathbb{E}_{y\sim p_Y}[\|G(y) - y\|_1] + \mathbb{E}_{x\sim p_X}[\|F(x) - x\|_1].$$

### 5.4 Total objective

$$\min_{G,F}\max_{D_X,D_Y}\;
\mathcal{L}_{GAN}(G, D_Y, X, Y) + \mathcal{L}_{GAN}(F, D_X, Y, X)
+ \lambda\,\mathcal{L}_{cyc}(G,F) + \gamma\,\mathcal{L}_{id}(G,F).$$

- $\lambda$ controls how strongly we enforce cycle consistency.
- $\gamma$ controls identity preservation (often smaller or sometimes omitted).


---

# Worked toy example (executable): CycleGAN on 2D point clouds

We’ll build a small CycleGAN that translates between two 2D domains **without pairs**:

- Domain X: points on a **ring** (circle-like)
- Domain Y: points on a **square ring** (square-like)

We will learn:
- $G$: ring → square-ish
- $F$: square-ish → ring

This runs fast on CPU and gives a clean visual intuition.


In [ ]:

# If needed in Colab:
# !pip -q install torch

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 6) Create unpaired datasets X and Y

We create two samplers:
- $x \sim p_X$: noisy circle/ring
- $y \sim p_Y$: noisy square ring

No pairing: we sample $x$ and $y$ independently.


In [ ]:

def sample_ring(n, r=2.0, noise=0.05):
    theta = np.random.rand(n) * 2*np.pi
    rad = r + np.random.randn(n) * (r*0.05)
    x = np.stack([rad*np.cos(theta), rad*np.sin(theta)], axis=1)
    x += noise*np.random.randn(n, 2)
    return x.astype(np.float32)

def sample_square_ring(n, r=2.0, noise=0.05):
    # Sample uniformly on square perimeter with small noise
    side = np.random.randint(0, 4, size=n)
    t = np.random.rand(n) * 2*r - r  # in [-r, r]
    x = np.zeros((n,2), dtype=np.float32)
    # top
    m = side == 0
    x[m,0] = t[m]; x[m,1] = r
    # bottom
    m = side == 1
    x[m,0] = t[m]; x[m,1] = -r
    # right
    m = side == 2
    x[m,0] = r; x[m,1] = t[m]
    # left
    m = side == 3
    x[m,0] = -r; x[m,1] = t[m]
    x += noise*np.random.randn(n,2).astype(np.float32)
    return x

X = sample_ring(2000)
Y = sample_square_ring(2000)

plt.figure()
plt.scatter(X[:,0], X[:,1], s=5, label="Domain X (ring)")
plt.scatter(Y[:,0], Y[:,1], s=5, label="Domain Y (square ring)")
plt.axis("equal"); plt.legend(); plt.title("Real unpaired samples from two domains")
plt.show()


## 7) Define networks: two generators and two discriminators

We use small MLPs:
- $G$: 2D → 2D (maps X to Y)
- $F$: 2D → 2D (maps Y to X)
- $D_X$: 2D → probability real in X
- $D_Y$: 2D → probability real in Y

Discriminators end with a sigmoid.


In [ ]:

class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x):
        return self.net(x)

class Disc(nn.Module):
    def __init__(self, in_dim=2, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

G = MLP(2, 2).to(device)  # X->Y
F = MLP(2, 2).to(device)  # Y->X
D_X = Disc(2).to(device)
D_Y = Disc(2).to(device)


## 8) Loss functions

We implement:
- Discriminator loss: maximize $\log D(\text{real}) + \log(1-D(\text{fake}))$
- Generator adversarial loss (non-saturating): minimize $-\log D(\text{fake})$
- Cycle loss: L1 distance


In [ ]:

eps = 1e-8

def d_loss(d_real, d_fake):
    return -(torch.log(d_real + eps).mean() + torch.log(1 - d_fake + eps).mean())

def g_adv_loss(d_fake):
    return -torch.log(d_fake + eps).mean()

l1 = nn.L1Loss()


## 9) Training loop (alternating updates)

We alternate updates:
- Update $D_X$ using real $x$ and fake $\hat{x}=F(y)$
- Update $D_Y$ using real $y$ and fake $\hat{y}=G(x)$
- Update generators $G$ and $F$ using adversarial + cycle loss

Generator total loss:

$$\ell_G = \ell_{adv}(G) + \ell_{adv}(F) + \lambda\,\ell_{cyc}.$$


In [ ]:

opt_DX = optim.Adam(D_X.parameters(), lr=1e-3, betas=(0.5, 0.9))
opt_DY = optim.Adam(D_Y.parameters(), lr=1e-3, betas=(0.5, 0.9))
opt_G  = optim.Adam(list(G.parameters()) + list(F.parameters()), lr=1e-3, betas=(0.5, 0.9))

def to_torch(a):
    return torch.tensor(a, device=device)

@torch.no_grad()
def plot_translation(step, n=2000):
    Xs = sample_ring(n)
    Ys = sample_square_ring(n)
    x = to_torch(Xs)
    y = to_torch(Ys)
    y_hat = G(x).cpu().numpy()
    x_hat = F(y).cpu().numpy()

    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.scatter(Xs[:,0], Xs[:,1], s=5, label="real X")
    plt.scatter(y_hat[:,0], y_hat[:,1], s=5, label="G(X) → fake Y")
    plt.axis("equal"); plt.legend(); plt.title(f"X and G(X) at step {step}")

    plt.subplot(1,2,2)
    plt.scatter(Ys[:,0], Ys[:,1], s=5, label="real Y")
    plt.scatter(x_hat[:,0], x_hat[:,1], s=5, label="F(Y) → fake X")
    plt.axis("equal"); plt.legend(); plt.title(f"Y and F(Y) at step {step}")
    plt.show()

steps = 4000
batch = 256
lambda_cyc = 10.0

DX_losses, DY_losses, G_losses, cyc_losses = [], [], [], []

for step in range(1, steps+1):
    x = to_torch(sample_ring(batch))
    y = to_torch(sample_square_ring(batch))

    # ---- Train D_X ----
    with torch.no_grad():
        x_fake = F(y)
    loss_DX = d_loss(D_X(x), D_X(x_fake))
    opt_DX.zero_grad()
    loss_DX.backward()
    opt_DX.step()

    # ---- Train D_Y ----
    with torch.no_grad():
        y_fake = G(x)
    loss_DY = d_loss(D_Y(y), D_Y(y_fake))
    opt_DY.zero_grad()
    loss_DY.backward()
    opt_DY.step()

    # ---- Train G and F ----
    y_fake = G(x)
    x_fake = F(y)

    loss_G_adv = g_adv_loss(D_Y(y_fake))
    loss_F_adv = g_adv_loss(D_X(x_fake))

    x_cyc = F(y_fake)
    y_cyc = G(x_fake)
    loss_cyc = l1(x_cyc, x) + l1(y_cyc, y)

    loss_total = loss_G_adv + loss_F_adv + lambda_cyc * loss_cyc

    opt_G.zero_grad()
    loss_total.backward()
    opt_G.step()

    DX_losses.append(loss_DX.item())
    DY_losses.append(loss_DY.item())
    G_losses.append(loss_total.item())
    cyc_losses.append(loss_cyc.item())

    if step in [1, 200, 800, 1500, 3000, 4000]:
        plot_translation(step)


## 10) Plot losses

Loss curves can be noisy; visuals matter more.


In [ ]:

plt.figure(figsize=(10,4))
plt.plot(DX_losses, label="D_X loss")
plt.plot(DY_losses, label="D_Y loss")
plt.title("Discriminator losses")
plt.xlabel("step"); plt.ylabel("loss")
plt.legend()
plt.show()

plt.figure(figsize=(10,4))
plt.plot(G_losses, label="Generator total loss")
plt.plot(cyc_losses, label="Cycle loss (unweighted)")
plt.title("Generator + cycle losses")
plt.xlabel("step"); plt.ylabel("loss")
plt.legend()
plt.show()


## 11) What to notice

If training worked:
- $G(X)$ should form a **square ring**.
- $F(Y)$ should form a **circular ring**.
- Cycle loss should stay reasonably low, meaning round-trip consistency.

This is the same principle used in image domain translation.


## 12) Quick recap

- Two generators: $G: X\to Y$, $F: Y\to X$
- Two discriminators: $D_Y$ (real/fake in Y), $D_X$ (real/fake in X)
- Adversarial loss enforces domain realism
- Cycle consistency enforces content preservation
- Identity loss (optional) discourages unnecessary changes
